# DIAMOND: Sterowanie Trajektorią w Celu Usunięcia Artefaktów (Flow Matching)

*Materiały na podstawie artykułu naukowego opracowanego przez badaczy z GMUM i IDEAS NCBR.*
> Link do artykułu: [arXiv:2602.00883](https://arxiv.org/pdf/2602.00883)
> Repozytorium GitHub: [gmum/DIAMOND](https://github.com/gmum/DIAMOND)

### [ZDECYDUJ: Tutaj wklej grafikę wprowadzającą - polecam Figurę 1 z artykułu pokazującą korekcję trajektorii z usunięciem artefaktów (np. błędnej dłoni)]
*(Rekomendowany znacznik to `![DIAMOND Wprowadzenie](img/fig1.png)`)*

---

Najnowsze modele generatywne tworzące obrazy z tekstu (takie jak FLUX czy SDXL) osiągnęły niesamowity poziom fotorealizmu. Niestety, wciąż borykają się z denerwującym problemem: **anatomicznymi i strukturalnymi artefaktami** (np. wygenerowanie sześciu palców u dłoni, połamane kończyny). 

Dotychczasowe metody naprawy tych błędów były inwazyjne – wymagały kosztownego dotrenowywania modeli (LoRA) albo ciężkich obliczeniowo poprawek już po wygenerowaniu obrazu (*post-hoc*).

Odpowiedzią na te problemy jest **DIAMOND** (*Directed Inference for Artifact Mitigation in Flow Matching Models*). Nie jest to nowy model generatywny, ale **innowacyjny mechanizm "kierowania" (guidance) nakładany w trakcie samej generacji obrazu (inference-time)**. DIAMOND działa w trybie *zero-shot* – całkowicie bez treningu i bez zmiany wag oryginalnego modelu!

Jak to możliwe? Model cały czas "zagląda w przyszłość", prognozuje, jak będzie wyglądał końcowy obraz wolny od szumu, wykrywa na nim nadchodzące błędy za pomocą detektora artefaktów, a następnie aktywnie i delikatnie "odpycha" główną trajektorię oddrogi prowadzącej do tych błędów.

In [ ]:
# Uruchom najpierw komórkę by sklonować paczkę z github (jeśli jej jeszcze nie posiadamy) i zainstalować ew. dodatki
!git clone https://github.com/gmum/DIAMOND
%cd DIAMOND
# tu możesz dodać %pip install -r requirements.txt lub inne kroki instalacyjne

## Jak fizycznie działa DIAMOND?

DIAMOND działa w procesie odszumiania na zasadzie ciągłej oceny czystości obrazu (korekcja trajektorii krok po kroku). Nie czeka on do momentu wygenerowania gotowego obrazu by go naprawić.

Proces krok po kroku przebiega następująco:

1. **Odtworzenie Czystego Obrazu "W Locie"**: Z modelu (szczególnie tych opartych o *Rectified Flow* takich jak FLUX, gdzie trajektorie są możliwie jak najprostsze/linearne) wyciągamy predykcję prędkości wektora (*velocity*). Model oblicza, jak mógłby wyglądać zdekodowany i całkowicie wolny od szumu obraz docelowy $\hat{x}_{0,t}$ w aktualnym kroku $t$.
2. **Detekcja Błędu (Artifact Detector)**: Szacowany czysty obraz przekazujemy do specjalnej, zamrożonej, pre-trenowanej i niezwykle ważnej części systemu: Detektora Artefaktów (Artifact Detector). Sieć ta skanuje obraz i weryfikuje za pomocą stworzonej maski, w którym miejscach pikseli powstanie artefakt (np. dodatkowy palec).
3. **Obliczenie Błędu (Loss) i Gradienty**: Skoro Detektor jest różniczkowalny (*differentiable*), system jest w stanie policzyć *pixel-wise loss* (stratę pikselową/segmentacyjną) na masce wytyczonej przez wadliwy obszar. Dzięki temu mechanizm tworzy wektor gradientu o tym jak bardzo (z jakim natężeniem) obraz musi się zmienić, by zminimalizować na nim ilość błędu.
4. **Odpychanie Trajektorii**: Otrzymane gradienty używamy bezpośrednio i "odpychamy" trajektorię na chwilę od miejsca błędu w samej przestrzeni ukrytej (latent), i posuwamy do przodu o ułamek sekundy $\Delta_t$ w tym unikalnym, korygującym kierunku. Następnie zmniejsza się waga skali tej poprawy - *dynamic scheduling*.


> *Dla przypomnienia z poprzedniej części: Dyfuzja oparta o tzw. Flow-Matching pozwala łączyć startowy próbkowany szum $x_1$ i wygenerowany ostatecznie obraz $x_0$ całkowicie liniowymi drogami. Sprawia to, że możemy wyliczyć predykcję końcową szybciej i prościej.*

### [ZDECYDUJ: Tutaj wklej grafikę potoku/rozwiązania DIAMOND pokazującego Artifact Detector i gradient - Figure 3 z papieru]
*(Rekomendowany znacznik to `![Flow Klatek](img/fig3.png)`)*

### Matematyka w pigułce: Linearne przesuwanie drogi (Rectified Flow)

Istotą Flow Matchingu jest to, że staramy się uczyć ścieżek transportujących dane z szumu $x_1$ do obrazu docelowego $x_0$ prostymi liniami:
$$x_t = (1 - t)x_0 + t x_1$$
Dzięki relacjom matematycznym tej ścieżki i tzw. "prędkości" $v_\theta(x_t, t)$, model przewiduje docelowy, czysty obraz (clean data latent) w każdym dowolnym kroku czasowym podczas tworzenia go na samym "żywym" szumie:

$$\hat{x}_{0,t} = x_t - t \cdot v_\theta(x_t, t)$$

Naszą sztuką "naprawy w locie" jest odświeżanie głównej zasady integracji Eulera, dodając do niej wektor kierunkowy przeskalowany odpowiednim suwakiem skali $s_t$:

$$ x_{t-\Delta t} = x_t - \Delta t \cdot \left[ v_\theta(x_t,t) + s_t \cdot \nabla_{x_t}\mathcal{L}_{artifact}(\hat{x}_{0,t}) \right]$$

To małe sprytne obejście powala "urwać" trajektorii dodatkowy ruch z gradientem w stronę eliminacji pikselowych artefaktów. Model nie traci na zdolnościach zapamiętanych podczas swojego oryginalnego trenowania. Oszczędzamy kosztowny prąd i dziesiątki godzin re-treningów!

---

---

### Zadanie [Obowiązkowe] - Odtworzenie korekcji pojedynczego kroku trajektorii w DIAMOND
Wiesz już, że przewagą modelu DIAMOND jest zdolność do naprawy trajektorii w jej trakcie (inference-time) przy użyciu mechanizmu *Artifact Detector* (Detektor Artefaktów). System prognozuje końcowy wynik na danym kroku $t$, weryfikuje jego dokładność i odpycha generację od niechcianych defektów obliczając spadki (gradienty).

Twój cel to ułożenie prostego szablonu jednej iteracji aktualizującej trajektorie obrazka (z wektora $\hat{x}_{0,t}$). Musisz wypełnić brakujące części algorytmu (miejsca oznaczone jako `???`) w funkcji `step_diamond`.

**Co musisz zrobić?**
1. Odtworzyć szacowany obraz $\hat{x}_{0,t}$ za pomocą udostępnionego wektora wyjściowego i odjętego fragmentu przebytego *czasu* i zmierzonej prędkosci $\dots - t \cdot v_\theta(x_t,t)$.
2. Ocenić artefaktową "szkodliwość" tego czystego obrazka używająć specjalnie przekierowanego modułu `artifact_detector`.
3. Dodaj gradient naprawczy `grad_x` w krok relokujący zmienną stanu (metoda odejmowania ułamkowego *dt* i odrzucająca dany artefaktowy wskaźnik) do `x_t_minus_dt`.


In [ ]:
import torch
import torch.nn as nn
# To jest uproszczony pseudokod ilustrujący pojedynczy krok wnioskowania w modelu DIAMOND

class DummyArtifactDetector(nn.Module):
    def forward(self, clean_image_estimate):
        # Symulacja błędu (Loss) dla artefaktów (np. 6 palców) na czystym obrazie
        # Im wyższa wartość funkcji straty, tym gorzej dla wygenerowanego obrazu
        loss = torch.sum(clean_image_estimate ** 2) 
        return loss

class DummyFlowModel:
    def __init__(self):
        self.artifact_detector = DummyArtifactDetector()
        
    def get_velocity(self, x_t, t):
        # Przewidywanie 'prędkości' u trajektorii modelu przez sieć układu (Flow Matching)
        # v_theta(x_t, t)
        return torch.randn_like(x_t)
        
    def step_diamond(self, x_t, t, dt, guidance_scale=1.0):
        # 1. Wymagamy by środowisko śledziło gradienty względem x_t (potrzebne do odpychania trajektorii)
        x_t = x_t.clone().detach().requires_grad_(True)
        
        # 2. Predykcja wektora prędkości v_theta(x_t, t) z zamrożonego wagi (bazowego modelu przepływu)
        v = self.get_velocity(x_t, t)
        
        # 3. Wytworzenie predykcji CZYSYTEGO obrazu (tzw. clean image estimate) na kroku czasu t
        # Wzór matematyczny: x_estimate = x_t - t * v
        x_estimate = x_t - t * v
        
        # 4. Ewaluacja błędu punktowego (Artifact Loss) na naszym "zapoznanym", czystym obrazie x_estimate
        # Musisz wrzucić czysty obraz w przygotowany detektor artefaktów by odkrył nieprawidłowości (zwrócił błąd loss)
        loss = self.artifact_detector(x_estimate)
        
        # 5. Obliczenie odpychającego gradientu (wektor odsuwający nas od wykrytego artefaktu)
        grad_x = torch.autograd.grad(loss, x_t)[0]
        
        # Sekretem metodologii DIAMOND jest normalizacja gradientu! 
        # Niezależnie czy robimy mały czy wielki poprawiany detal, kierunek pozostanie niezaburzony.
        grad_norm = grad_x / (torch.norm(grad_x) + 1e-8)
        
        # 6. Aktualizacja trajektorii modelu (Linear- korygujący krok modyfikowany o znormalizowany gradient naprawczy DIAMOND z siłą guidance_scale)
        # Wzór: x_{t - dt} = x_t - dt * ( v + s_t * \nabla_x Loss / ||\nabla_x Loss|| )
        x_t_minus_dt = x_t - dt * ( v + guidance_scale * grad_norm )
        
        return x_t_minus_dt.detach()

# --- Inicjalizacja i wykonanie instrukcji w teście ---
sim_model = DummyFlowModel()

# Losowy szum stanu obrazu (Latent x_t) dany w czasie t=0.5 z modelu generatywnego
current_latent_x_t = torch.randn(1, 4, 64, 64) 
t_current = 0.5
time_step_dt = 0.05
scale_of_correction = 5.0 # (s_t) - siła naprawcza detektora DIAMOND (guidance scale)

# Wykonaj zaktualizowany jednorazowy krok naprawczy DIAMOND na przebytej trajektorii
next_latent_step = sim_model.step_diamond(
    x_t=current_latent_x_t, 
    t=t_current, 
    dt=time_step_dt, 
    guidance_scale=scale_of_correction
)

print("Kolejna iteracja ukrytego 'czystego' po zredukowaniu krokem z korektą DIAMOND wykonana! Wymiary:", next_latent_step.shape)

---

## Kluczowe Cechy Metodologii DIAMOND

Aby powyższy proces działał stabilnie w rzeczywistości na milionach parametrów, autorzy DIAMOND wdrożyli dwie krytyczne optymalizacje opisane w sekcji metodologii:

1. **Normalizacja Gradientu (Gradient Normalization):**
Naiwne odjęcie wektora $\nabla_{x_t} \mathcal{L}_a$ niesie ukryte ryzyko. Wielkość (norma) gradientu bardzo rośnie, jeśli na docelowym obrazku znajduje się wiele artefaktów lub sam artefakt jest duży, mimo że potrzebujemy *tylko znać kierunek* zmiany, a nie gwałtownie przerzucać obrazek.
Zabieg optymalizujący sprowadza się do usunięcia "siły" gradientu za pomocą stałego wektora:
$$ \delta_t = \lambda_t \frac{\nabla_{x_t} L_a}{||\nabla_{x_t} L_a||_2 + \epsilon} $$
Oddzielenie siły (określanej od razu przez skalar przypisany do kroku generacji $\lambda_t$) od kierunku gradientu upewnia nas w tym, że proces zawsze pozostanie spójny, nieważne czy łatamy mały paznokieć czy cały dziwny kontur drugiego nosa.

2. **Skalowanie w czasie (Power Schedule, $\lambda_t$):**
Podczas pracy nad każdym krokiem w dyfuzji, początkowo generowany jest "brudny zarys" kształtów globalnych (**coarse features** z ang.), a dopieszczanie detali (włosy, cienie, ostrości) zachodzą blisko końca tworzenia (**high-frequency details**).
Artefakty takie jak "dziwne ułożenie palców", "dodatkowa liczba kończyn" lub "zepsute litery" są problemem strukturalnym - rodzą się w fazie zarysów! Zależność parametrów naprawy DIAMOND ustala się za pomocą funkcji wielomianowej ($Power Schedule$). Siła guidance jest gigantyczna na początku dyfuzji (kiedy generowane jest ułożenie całego ciała) a następnie znacząco wygasa. Kiedy model dopieszcza końcowe cieniowanie, nie musimy go już odpychać.


### Zdumiewające wyniki (Ablation Studies & Empirical Studies)

Zespół przeprowadził szereg testów, badając, jak wprowadzenie DIAMONDu naprawia artefakty na modelach *FLUX.1*, *FLUX.2* oraz starszym *SDXL*. Testowali swoje rozwiązanie porównując je metrykami na bazach danych o "tekstach" (words) i "ludzkiej anatomii" (people).

* **Mean Artifact Frequency (Częstotliwość Błędów):** W teście na pisaniu słów FLUX.1 generował ucięcia i krzywe litery aż w 100% przypadkach. Aplikacja DIAMONDa natychmiastowo zmiotła błędy w zaledwie do ok. 10%. Dla zwierząt częstotliwość potrafiła spaść 40% $\rightarrow$ do około 30%.
* **Zachowanie spójności Promtpu (CLIP-T score):** Model z aktywnym i rygorystycznym układem "odpychania" mógłby wygenerować coś zupełnie niepożądanego w stosunku do tego o co poprosiliśmy z tekstu. Nic bardziej mylnego - we wszystkich testach **CLIP-T** pozostał praktycznie nietknięty, a w wielu razach wręcz znacznie wzrósł (bo przecież dostał poprawioną strukturę).
* **Całkowita ochrona bezbłędnej reszty obrazka (MAE Non-Artifact):** To kluczowa zmierzona przewaga nad post-procesowymi modelami takimi jak *HandsXL*. Modele dotrenowane często wariowały na innych elementach (lub generowały "mydełko" detali). DIAMOND miał na pikselach bez artefaktów (Non-Artifact MAE) pomijalną różnicę, dumnie zachowując autorską kompozycję nieistotnych regionów tła np. na poziomie $0.009$.

> *Wniosek: Nie warto dotrenowywać modeli (LoRA) i ryzykować spalenie pozostałej wyuczonej części modelu bazowego! Jeżeli potrzebujesz perfekcyjnej anatomii, użyj modelu detekcji połączonego wektorowo z wbudowanym wnioskowaniem przepływu (Flow Matching).*